## （１）国土数値情報 駅別乗降客数データ (S12-25)

In [1]:
import zipfile
from pathlib import Path

import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", None)

# このノートブックは script/ に置かれている想定。リポジトリ直下を基準にする。
BASE_DIR = Path.cwd().parent if Path.cwd().name == "script" else Path.cwd()
ZIP_PATH = BASE_DIR / "data" / "駅別乗降客数データ" / "S12-25_GML.zip"
assert ZIP_PATH.exists(), f"ZIP が見つかりません: {ZIP_PATH}"

In [2]:
# ZIP 内のファイル一覧を確認
with zipfile.ZipFile(ZIP_PATH) as zf:
    for name in zf.namelist():
        print(name)

S12-25_GML/
S12-25_GML/KS-META-S12-25.xml
S12-25_GML/KsjAppSchema-S12-v3_3.xsd
S12-25_GML/Shift-JIS/
S12-25_GML/Shift-JIS/S12-25.xml
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.dbf
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.geojson
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.prj
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.shp
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.shx
S12-25_GML/UTF-8/
S12-25_GML/UTF-8/S12-25.xml
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.dbf
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.geojson
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.prj
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.shp
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.shx


In [3]:
# ZIP を解凍せず、geopandas の zip:// 仮想パスで直接 GeoJSON を読み込む
GEOJSON_INNER = "S12-25_GML/UTF-8/S12-25_NumberOfPassengers.geojson"
read_path = f"zip://{ZIP_PATH}!{GEOJSON_INNER}"

gdf = gpd.read_file(read_path)
print("行数 x 列数:", gdf.shape)
print("CRS:", gdf.crs)

行数 x 列数: (10534, 64)
CRS: EPSG:6668


In [4]:
# 属性列を S12 仕様の日本語名にリネーム
# 固定項目
rename_map = {
    "S12_001": "駅名",
    "S12_001c": "駅コード",
    "S12_001g": "グループコード",
    "S12_002": "運営会社",
    "S12_003": "路線名",
    "S12_004": "鉄道区分",
    "S12_005": "事業者種別",
}

# 年度ごとに [重複コード, データ有無コード, 備考, 乗降客数] が4列ずつ繰り返す
# 2011年度=S12_006〜S12_009 ... 2024年度=S12_058〜S12_061
for i, year in enumerate(range(2011, 2025)):
    base = 6 + i * 4
    rename_map[f"S12_{base:03d}"] = f"重複コード{year}"
    rename_map[f"S12_{base + 1:03d}"] = f"データ有無コード{year}"
    rename_map[f"S12_{base + 2:03d}"] = f"備考{year}"
    rename_map[f"S12_{base + 3:03d}"] = f"乗降客数{year}"

gdf = gdf.rename(columns=rename_map)
list(gdf.columns)

['駅名',
 '駅コード',
 'グループコード',
 '運営会社',
 '路線名',
 '鉄道区分',
 '事業者種別',
 '重複コード2011',
 'データ有無コード2011',
 '備考2011',
 '乗降客数2011',
 '重複コード2012',
 'データ有無コード2012',
 '備考2012',
 '乗降客数2012',
 '重複コード2013',
 'データ有無コード2013',
 '備考2013',
 '乗降客数2013',
 '重複コード2014',
 'データ有無コード2014',
 '備考2014',
 '乗降客数2014',
 '重複コード2015',
 'データ有無コード2015',
 '備考2015',
 '乗降客数2015',
 '重複コード2016',
 'データ有無コード2016',
 '備考2016',
 '乗降客数2016',
 '重複コード2017',
 'データ有無コード2017',
 '備考2017',
 '乗降客数2017',
 '重複コード2018',
 'データ有無コード2018',
 '備考2018',
 '乗降客数2018',
 '重複コード2019',
 'データ有無コード2019',
 '備考2019',
 '乗降客数2019',
 '重複コード2020',
 'データ有無コード2020',
 '備考2020',
 '乗降客数2020',
 '重複コード2021',
 'データ有無コード2021',
 '備考2021',
 '乗降客数2021',
 '重複コード2022',
 'データ有無コード2022',
 '備考2022',
 '乗降客数2022',
 '重複コード2023',
 'データ有無コード2023',
 '備考2023',
 '乗降客数2023',
 '重複コード2024',
 'データ有無コード2024',
 '備考2024',
 '乗降客数2024',
 'geometry']

In [5]:
gdf.info()
gdf.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 10534 entries, 0 to 10533
Data columns (total 64 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   駅名            10534 non-null  object  
 1   駅コード          10297 non-null  object  
 2   グループコード       10297 non-null  object  
 3   運営会社          10534 non-null  object  
 4   路線名           10534 non-null  object  
 5   鉄道区分          10534 non-null  int32   
 6   事業者種別         10534 non-null  int32   
 7   重複コード2011     10534 non-null  int32   
 8   データ有無コード2011  10534 non-null  int32   
 9   備考2011        122 non-null    object  
 10  乗降客数2011      10534 non-null  int32   
 11  重複コード2012     10534 non-null  int32   
 12  データ有無コード2012  10534 non-null  int32   
 13  備考2012        144 non-null    object  
 14  乗降客数2012      10534 non-null  int32   
 15  重複コード2013     10534 non-null  int32   
 16  データ有無コード2013  10534 non-null  int32   
 17  備考2013        159 non-null    object  
 18

,駅名,駅コード,グループコード,運営会社,路線名,鉄道区分,事業者種別,重複コード2011,データ有無コード2011,備考2011,乗降客数2011,重複コード2012,データ有無コード2012,備考2012,乗降客数2012,重複コード2013,データ有無コード2013,備考2013,乗降客数2013,重複コード2014,データ有無コード2014,備考2014,乗降客数2014,重複コード2015,データ有無コード2015,備考2015,乗降客数2015,重複コード2016,データ有無コード2016,備考2016,乗降客数2016,重複コード2017,データ有無コード2017,備考2017,乗降客数2017,重複コード2018,データ有無コード2018,備考2018,乗降客数2018,重複コード2019,データ有無コード2019,備考2019,乗降客数2019,重複コード2020,データ有無コード2020,備考2020,乗降客数2020,重複コード2021,データ有無コード2021,備考2021,乗降客数2021,重複コード2022,データ有無コード2022,備考2022,乗降客数2022,重複コード2023,データ有無コード2023,備考2023,乗降客数2023,重複コード2024,データ有無コード2024,備考2024,乗降客数2024,geometry
0,二月田,010112,010112,九州旅客鉄道,指宿枕崎線,11,2,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,1,None,658,1,1,None,690,1,1,None,318,1,1,None,305,1,1,None,622,1,1,None,664,1,1,None,726,"LINESTRING (130.63035 31.25405, 130.62985 31.2..."
1,古島,010127,010127,沖縄都市モノレール,沖縄都市モノレール線,23,5,1,1,None,3907,1,1,None,3980,1,1,None,4572,1,1,None,4187,1,1,None,4648,1,1,None,4528,1,1,None,4819,1,1,None,5086,1,1,None,5143,1,1,None,3404,1,1,None,3534,1,1,None,4401,1,1,None,4903,1,1,None,5490,"LINESTRING (127.70279 26.23035, 127.70309 26.2..."
2,お台場海浜公園,004091,004091,ゆりかもめ,東京臨海新交通臨海線,24,5,1,1,None,14612,1,1,None,16130,1,1,None,16357,1,1,None,16515,1,1,None,17537,1,1,None,17488,1,1,None,16944,1,1,None,17165,1,1,None,16899,1,1,None,7703,1,1,None,8535,1,1,None,11171,1,1,None,13195,1,1,None,13435,"LINESTRING (139.77818 35.62961, 139.77888 35.63)"
3,東京国際クルーズターミナル,004128,004128,ゆりかもめ,東京臨海新交通臨海線,24,5,1,1,None,3767,1,1,None,3235,1,1,None,3407,1,1,None,3782,1,1,None,3682,1,1,None,3682,1,1,None,3590,1,1,None,3551,1,1,None,3191,1,1,None,1383,1,1,None,1984,1,1,None,2300,1,1,None,2963,1,1,None,3503,"LINESTRING (139.77333 35.62109, 139.77288 35.6..."
4,テレコムセンター,004144,004144,ゆりかもめ,東京臨海新交通臨海線,24,5,1,1,None,12112,1,1,None,12775,1,1,None,13366,1,1,None,14138,1,1,None,14069,1,1,None,14069,1,1,None,13744,1,1,None,13475,1,1,None,12140,1,1,None,6536,1,1,None,7631,1,1,None,8118,1,1,None,8505,1,1,None,9267,"LINESTRING (139.78001 35.61791, 139.77932 35.6..."


## （２）駅グループ単位への集約

`docs/passenger_aggregation.md` の設計に基づき、**駅名 + 半径1km** で駅をまとめ、
レベル時系列（各年の乗降客数）と増減率（パターン1・2）を生成する。

- 在/欠測の判定: `データ有無コード==1` を実測（値0も実測の0）、それ以外は欠測（0扱いしない）
- 構成単位: 運営会社（路線は事業者内で合算。重複行は0なので二重計上なし）
- 出力: メイン `station_group`（1群1行）＋ 監査用 `station_group_operator`（1群×1社）

In [6]:
# 集約のための定数と前処理
import numpy as np
from sklearn.cluster import DBSCAN

YEARS = list(range(2011, 2025))
PRE_WINDOW  = [2017, 2018, 2019]   # コロナ前の参照窓（平常年は代替可能）
POST_WINDOW = [2023, 2024]         # コロナ後（回復期）の参照窓。2022以前は底値なので不可

# 乗降客数列を数値化（在/欠測の判定は「データ有無コード」で行うため、ここは数値化のみ）
for y in YEARS:
    gdf[f"乗降客数{y}"] = pd.to_numeric(gdf[f"乗降客数{y}"], errors="coerce")

In [7]:
# 空間グループ: 代表点を取り、駅名ごとに haversine DBSCAN(eps=1km) でクラスタリング
_pt = gdf.geometry.representative_point()
gdf["lon"] = _pt.x
gdf["lat"] = _pt.y

EARTH_R = 6_371_000.0          # 地球半径[m]
EPS     = 1000 / EARTH_R       # 半径1km を弧度に換算

grp = pd.Series(index=gdf.index, dtype=object)
for name, sub_idx in gdf.groupby("駅名").groups.items():
    sub_idx = list(sub_idx)
    if len(sub_idx) == 1:                       # 単独駅はクラスタリング不要
        grp.loc[sub_idx[0]] = f"{name}#0"
        continue
    coords = np.radians(gdf.loc[sub_idx, ["lat", "lon"]].to_numpy())
    lab = DBSCAN(eps=EPS, min_samples=1, metric="haversine").fit(coords).labels_
    for j, l in zip(sub_idx, lab):
        grp.loc[j] = f"{name}#{l}"
gdf["grp"] = grp   # 例: "東京#0"（駅名を内包。同名遠方駅は #1, #2 ... に分かれる）

print("駅グループ数:", gdf["grp"].nunique(), " / 元の行数:", len(gdf))

駅グループ数: 9273  / 元の行数: 10534


In [8]:
# (grp, 運営会社, 年) に集約 → 運営会社×年 のワイド表（present / pax）
_rows = []
for y in YEARS:
    g = (gdf.groupby(["grp", "運営会社"])
            .agg(present=(f"データ有無コード{y}", lambda s: bool((s == 1).any())),
                 pax=(f"乗降客数{y}", "sum"))                  # 重複行は0なので二重計上なし
            .reset_index())
    g["year"] = y
    _rows.append(g)
op_year = pd.concat(_rows, ignore_index=True)

pres = (op_year.pivot_table(index=["grp", "運営会社"], columns="year",
                            values="present", aggfunc="first").fillna(False)[YEARS])
paxw = (op_year.pivot_table(index=["grp", "運営会社"], columns="year",
                            values="pax", aggfunc="first").fillna(0.0)[YEARS])
print("運営会社×群:", len(pres))

運営会社×群: 9781


### 指標① レベル時系列（inclusive・補間なし）

その年に実測のある全社を合算。欠測は0扱いせず、全社欠測の年は NaN（グラフは線を切る）。
社が抜けた年は寄与社数 `n_op` が下がるので、谷が「データ穴」か「実減」か判別できる。

In [9]:
# 各年の合算（在の社のみ）と、群の延べ社数・構成一定フラグ
present_long = op_year[op_year["present"]]

level = (present_long.groupby(["grp", "year"])
         .agg(pax=("pax", "sum"))
         .reset_index())
pax_wide = level.pivot(index="grp", columns="year", values="pax").reindex(columns=YEARS)
pax_wide.columns = [f"pax_{y}" for y in YEARS]   # 欠測年は自然に NaN

# 群の延べ運営会社数
n_op = pres.reset_index().groupby("grp")["運営会社"].nunique()

# level_complete: 構成が全データ年で一定か
#   = 群に現れる全社が、群がデータを持つ全年で在 であること
op_present_years = present_long.groupby(["grp", "運営会社"])["year"].nunique()
grp_data_years   = present_long.groupby("grp")["year"].nunique()
min_op_years     = op_present_years.groupby("grp").min()
level_complete   = (min_op_years == grp_data_years.reindex(min_op_years.index))

print("構成一定の群:", int(level_complete.sum()), "/", len(level_complete))

構成一定の群: 8434 / 8598


### 指標② パターン1（直近 2024/2023）

2023・2024の両方で実測があり、2023値>0 の社だけを basket に合算して比率を取る。

In [10]:
# パターン1: rate_yoy と異常値フラグ
m1 = pres[2023] & pres[2024] & (paxw[2023] > 0)
d1 = paxw.loc[m1, 2023].groupby(level="grp").sum()
n1 = paxw.loc[m1, 2024].groupby(level="grp").sum()
rate_yoy = (n1 / d1 - 1.0)
rate_yoy = rate_yoy[d1 > 0]
flag_yoy = rate_yoy.abs() > 0.30          # 事業者内の報告基準変更などを検出

print("パターン1 算出可能:", len(rate_yoy), "群  中央値: {:+.2f}%".format(rate_yoy.median()*100))

パターン1 算出可能: 7367 群  中央値: +2.14%


### 指標③ パターン2（コロナ前→後・事業者単位）

各社について「pre窓(2017-19)の最新実測」→「post窓(2023-24)の最新実測」を取り、
両方そろう社だけを basket に合算。参照年は社ごとに異なってよい（各社は自分基準で like-for-like）。

In [11]:
# パターン2: rate_covid と信頼性フラグ
def _latest(years_win):
    """各(grp,運営会社)について、窓内で present かつ pax>0 の最新年と値を返す"""
    yr  = pd.Series(np.nan, index=pres.index)
    val = pd.Series(0.0,    index=pres.index)
    for y in years_win:                       # 昇順に上書き → 最大年が残る
        ok  = pres[y] & (paxw[y] > 0)
        yr  = yr.mask(ok, y)
        val = val.mask(ok, paxw[y])
    return yr, val

pre_year,  pre_val  = _latest(PRE_WINDOW)
post_year, post_val = _latest(POST_WINDOW)

basket2 = pre_year.notna() & post_year.notna()           # pre/post 両方ある社のみ（pre/post一貫）
d2 = pre_val[basket2].groupby(level="grp").sum()
n2 = post_val[basket2].groupby(level="grp").sum()
rate_covid = (n2 / d2 - 1.0)
rate_covid = rate_covid[d2 > 0]

# 品質: 被覆率(basket社数/延べ社数) と pre年の最古
n_basket  = basket2.groupby(level="grp").sum()
cov_covid = n_basket / n_op.reindex(n_basket.index)
min_pre   = pre_year[basket2].groupby(level="grp").min()
flag_covid = (((cov_covid < 1.0) | (min_pre < 2019)).reindex(rate_covid.index, fill_value=True)
              | (rate_covid.abs() > 1.0))   # 極端値(|率|>100%)も要注意（小駅の分母ノイズ等。rate_yoyと設計統一）

print("パターン2 算出可能:", len(rate_covid), "群  中央値: {:+.2f}%".format(rate_covid.median()*100))

パターン2 算出可能: 7684 群  中央値: -8.40%


### メイン / 監査用テーブルの組み立て

In [12]:
# メイン station_group（1群1行）。今後 人口・地価などはこの右に追記していく
ident = gdf.groupby("grp").agg(駅名=("駅名", "first"), lon=("lon", "mean"), lat=("lat", "mean"))

station_group = ident.join(pax_wide)
station_group["n_op"]           = n_op.reindex(station_group.index)
station_group["level_complete"] = level_complete.reindex(station_group.index, fill_value=True)
station_group["rate_yoy"]       = (rate_yoy * 100).round(1).reindex(station_group.index)
station_group["flag_yoy"]       = flag_yoy.reindex(station_group.index, fill_value=False)
station_group["rate_covid"]     = (rate_covid * 100).round(1).reindex(station_group.index)
station_group["flag_covid"]     = flag_covid.reindex(station_group.index, fill_value=False)

for y in YEARS:                                   # NaN許容整数に
    station_group[f"pax_{y}"] = station_group[f"pax_{y}"].round().astype("Int64")
station_group = station_group.reset_index()

print("station_group:", station_group.shape)
station_group.head()

station_group: (9273, 24)


,grp,駅名,lon,lat,pax_2011,pax_2012,pax_2013,pax_2014,pax_2015,pax_2016,pax_2017,pax_2018,pax_2019,pax_2020,pax_2021,pax_2022,pax_2023,pax_2024,n_op,level_complete,rate_yoy,flag_yoy,rate_covid,flag_covid
0,JA広島病院前#0,JA広島病院前,132.32251,34.34405,3403,3403,3403,3403,3403,3403,1674,1548,1474,1153,1085,1178,1270,1474,1,True,16.1,False,0.0,False
1,JR三山木#0,JR三山木,135.78427,34.79899,1484,1586,1613,1676,1800,1866,1899,2004,2138,1626,1678,1762,1924,2052,1,True,6.7,False,-4.0,False
2,JR五位堂#0,JR五位堂,135.71831,34.52768,1521,1493,1448,1444,1484,1526,1567,1576,1514,1246,1314,1364,1436,1424,1,True,-0.8,False,-5.9,False
3,JR俊徳道#0,JR俊徳道,135.57249,34.65830,6471,6899,7347,7512,7802,7948,8147,8394,10402,8108,9494,11170,12180,13006,1,True,6.8,False,25.0,False
4,JR小倉#0,JR小倉,135.78634,34.88913,3817,3828,3871,3816,3844,3724,3752,3734,3700,2828,2944,3282,3632,3992,1,True,9.9,False,7.9,False


In [13]:
# 監査用 station_group_operator（1群×1社）。各年値・在/欠測・参照年を保持
_pax = paxw.where(pres, np.nan)                    # 欠測は NaN（実測の0と区別）
_pax.columns  = [f"pax_{y}" for y in YEARS]
_present = pres.copy()
_present.columns = [f"present_{y}" for y in YEARS]

station_group_operator = _pax.join(_present)
station_group_operator["pre_year"]  = pre_year
station_group_operator["post_year"] = post_year
for y in YEARS:
    station_group_operator[f"pax_{y}"] = station_group_operator[f"pax_{y}"].round().astype("Int64")
station_group_operator = station_group_operator.reset_index()

print("station_group_operator:", station_group_operator.shape)
station_group_operator.head()

station_group_operator: (9781, 32)


,grp,運営会社,pax_2011,pax_2012,pax_2013,pax_2014,pax_2015,pax_2016,pax_2017,pax_2018,pax_2019,pax_2020,pax_2021,pax_2022,pax_2023,pax_2024,present_2011,present_2012,present_2013,present_2014,present_2015,present_2016,present_2017,present_2018,present_2019,present_2020,present_2021,present_2022,present_2023,present_2024,pre_year,post_year
0,JA広島病院前#0,広島電鉄,3403,3403,3403,3403,3403,3403,1674,1548,1474,1153,1085,1178,1270,1474,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0
1,JR三山木#0,西日本旅客鉄道,1484,1586,1613,1676,1800,1866,1899,2004,2138,1626,1678,1762,1924,2052,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0
2,JR五位堂#0,西日本旅客鉄道,1521,1493,1448,1444,1484,1526,1567,1576,1514,1246,1314,1364,1436,1424,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0
3,JR俊徳道#0,西日本旅客鉄道,6471,6899,7347,7512,7802,7948,8147,8394,10402,8108,9494,11170,12180,13006,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0
4,JR小倉#0,西日本旅客鉄道,3817,3828,3871,3816,3844,3724,3752,3734,3700,2828,2944,3282,3632,3992,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0


## （３）都道府県・駅名表記・検索名表記

国土数値情報「行政区域(N03)」の都道府県ポリゴンを群の代表点に空間結合（`gpd.sjoin`）して `都道府県` を付与し、表示用 `label` と検索用 `search_label` を生成する。

- **label（表示用）**: 単独駅名→`駅名` ／ 同名複数でも「複数運営会社」「同一社が別地域に衝突」は `駅名`（素）／ 単独運営会社で衝突なしは `駅名（運営会社）`
- **search_label（検索用）**: `label` に `都道府県` を付加（単一括弧）。例: 尼崎（阪神電気鉄道・兵庫県）／ 二月田（鹿児島県）／ 上道（鳥取県）
- `grp` は結合キーとして温存。`都道府県` も単独列で保持。

In [14]:
# 行政区域(N03)の都道府県ポリゴンを代表点に空間結合して 都道府県 を付与
N03_ZIP = BASE_DIR / "data" / "行政区域データ" / "N03-20260101_GML.zip"
# 都道府県名でまとまった _prefecture 版を使用。CRS は EPSG:6668 で S12 と一致（再投影不要）
pref = gpd.read_file(f"zip://{N03_ZIP}!N03-20260101_prefecture.shp", columns=["N03_001"])
pref = pref.rename(columns={"N03_001": "都道府県"})

grp_pt = gpd.GeoDataFrame(
    station_group[["grp"]].copy(),
    geometry=gpd.points_from_xy(station_group["lon"], station_group["lat"]),
    crs="EPSG:6668",
)
# 点が含まれる都道府県を判定。境界外（海上の埋立地・座標誤差）は最近傍で補完
hit = gpd.sjoin(grp_pt, pref, how="left", predicate="within").drop_duplicates("grp")[["grp", "都道府県"]]
miss = hit["都道府県"].isna()
if miss.any():
    near = (gpd.sjoin_nearest(grp_pt[grp_pt["grp"].isin(hit.loc[miss, "grp"])].to_crs(3857), pref.to_crs(3857), how="left")
            .drop_duplicates("grp").set_index("grp")["都道府県"])
    hit.loc[miss, "都道府県"] = hit.loc[miss, "grp"].map(near)

station_group = station_group.merge(hit, on="grp", how="left")
print("都道府県 付与:", int(station_group["都道府県"].notna().sum()), "/", len(station_group),
      " 未割当:", int(station_group["都道府県"].isna().sum()))
print(station_group["都道府県"].value_counts().head())

都道府県 付与: 9273 / 9273  未割当: 0
都道府県
東京都    654
北海道    558
大阪府    480
愛知県    477
兵庫県    365
Name: count, dtype: int64


In [15]:
# 表示用 label と 検索用 search_label を生成
ops_by_grp = (station_group_operator.groupby("grp")["運営会社"]
              .apply(lambda s: sorted(s.unique().tolist())).to_dict())
grps_by_name = station_group.groupby("駅名")["grp"].apply(list).to_dict()

def make_label(nm, grp):
    grps = grps_by_name[nm]
    if len(grps) == 1:                               # A: 単独駅名
        return nm
    ops = ops_by_grp.get(grp, [])
    if len(ops) != 1:                                # C: 複数運営会社 → 素
        return nm
    op = ops[0]
    for g in grps:                                   # D: 同一単独社が別地域に衝突 → 素
        if g != grp and ops_by_grp.get(g, []) == [op]:
            return nm
    return f"{nm}（{op}）"                             # B: 単独社・衝突なし → 社付き

station_group["label"] = [make_label(r["駅名"], r["grp"]) for _, r in station_group.iterrows()]

def make_search_label(label, pref):
    if pd.isna(pref):
        return label
    if label.endswith(f"（{pref}）"):                  # 運営会社名が都道府県名と同一（例: 都営=東京都）
        return label
    if label.endswith("）"):                          # 駅名（運営会社） → 駅名（運営会社・都道府県）
        return label[:-1] + "・" + pref + "）"
    return f"{label}（{pref}）"                         # 駅名 → 駅名（都道府県）

station_group["search_label"] = [make_search_label(l, p)
                                 for l, p in zip(station_group["label"], station_group["都道府県"])]

# 列順を整える（識別子を前方へ）
front = ["grp", "駅名", "label", "search_label", "都道府県", "lon", "lat"]
station_group = station_group[front + [c for c in station_group.columns if c not in front]]

# 検索ラベルの一意性チェック（将来データで同名・同県・素ラベルが衝突したら検知）
print("search_label 重複:", int(station_group["search_label"].duplicated().sum()), "（0 が正常）")
station_group.head()

search_label 重複: 0 （0 が正常）


,grp,駅名,label,search_label,都道府県,lon,lat,pax_2011,pax_2012,pax_2013,pax_2014,pax_2015,pax_2016,pax_2017,pax_2018,pax_2019,pax_2020,pax_2021,pax_2022,pax_2023,pax_2024,n_op,level_complete,rate_yoy,flag_yoy,rate_covid,flag_covid
0,JA広島病院前#0,JA広島病院前,JA広島病院前,JA広島病院前（広島県）,広島県,132.32251,34.34405,3403,3403,3403,3403,3403,3403,1674,1548,1474,1153,1085,1178,1270,1474,1,True,16.1,False,0.0,False
1,JR三山木#0,JR三山木,JR三山木,JR三山木（京都府）,京都府,135.78427,34.79899,1484,1586,1613,1676,1800,1866,1899,2004,2138,1626,1678,1762,1924,2052,1,True,6.7,False,-4.0,False
2,JR五位堂#0,JR五位堂,JR五位堂,JR五位堂（奈良県）,奈良県,135.71831,34.52768,1521,1493,1448,1444,1484,1526,1567,1576,1514,1246,1314,1364,1436,1424,1,True,-0.8,False,-5.9,False
3,JR俊徳道#0,JR俊徳道,JR俊徳道,JR俊徳道（大阪府）,大阪府,135.57249,34.65830,6471,6899,7347,7512,7802,7948,8147,8394,10402,8108,9494,11170,12180,13006,1,True,6.8,False,25.0,False
4,JR小倉#0,JR小倉,JR小倉,JR小倉（京都府）,京都府,135.78634,34.88913,3817,3828,3871,3816,3844,3724,3752,3734,3700,2828,2944,3282,3632,3992,1,True,9.9,False,7.9,False


In [16]:
# 確認: ラベル生成の代表例（A:単独 / B:社付き / C:複数社→素 / D:同一社衝突→素）
_chk = ["二月田#0", "尼崎#0", "尼崎#1", "市役所前#0", "上道#0", "上道#1",
        "大久保#2", "大久保#3", "三田#0", "三田#1"]
station_group[station_group["grp"].isin(_chk)][["grp", "label", "search_label", "都道府県"]]

,grp,label,search_label,都道府県
389,三田#0,三田,三田（兵庫県）,兵庫県
390,三田#1,三田（東京都）,三田（東京都）,東京都
572,上道#0,上道,上道（鳥取県）,鳥取県
573,上道#1,上道,上道（岡山県）,岡山県
982,二月田#0,二月田,二月田（鹿児島県）,鹿児島県
2701,大久保#2,大久保,大久保（秋田県）,秋田県
2702,大久保#3,大久保,大久保（東京都）,東京都
3512,尼崎#0,尼崎（阪神電気鉄道）,尼崎（阪神電気鉄道・兵庫県）,兵庫県
3513,尼崎#1,尼崎（西日本旅客鉄道）,尼崎（西日本旅客鉄道・兵庫県）,兵庫県
3830,市役所前#0,市役所前（伊予鉄道）,市役所前（伊予鉄道・愛媛県）,愛媛県


In [17]:
# 確認: 代表的な駅（東京・新宿・渋谷・横浜・小諸）
_check = ["東京#0", "新宿#0", "渋谷#0", "横浜#0", "小諸#0"]
_cols = ["grp", "駅名", "n_op", "level_complete",
         "pax_2019", "pax_2024", "rate_yoy", "flag_yoy", "rate_covid", "flag_covid"]
station_group[station_group["grp"].isin(_check)][_cols]

,grp,駅名,n_op,level_complete,pax_2019,pax_2024,rate_yoy,flag_yoy,rate_covid,flag_covid
3481,小諸#0,小諸,2,False,6518,5647,119.0,True,58.0,True
4376,新宿#0,新宿,5,True,3265570,2713386,-9.1,False,-16.9,True
4933,東京#0,東京,3,False,1339555,1262604,7.3,False,-5.7,False
5524,横浜#0,横浜,6,True,2101709,1936287,-1.3,False,-7.9,True
6039,渋谷#0,渋谷,4,True,2704384,2897703,36.1,True,7.1,False


In [18]:
# CSV 出力（geometry を持たない素の表。後続の結合・可視化で利用）
OUT_DIR = BASE_DIR / "data" / "derived"
OUT_DIR.mkdir(parents=True, exist_ok=True)
station_group.to_csv(OUT_DIR / "station_group.csv", index=False)
station_group_operator.to_csv(OUT_DIR / "station_group_operator.csv", index=False)
print("saved:", OUT_DIR / "station_group.csv")

saved: /Users/kensuke26/Desktop/AI_Native/01Project/dev_Database_Map/AI-Database-Map/data/derived/station_group.csv
